In [ ]:
import numpy as np
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.metrics import silhouette_score
from itertools import product
import torch
from bertopic.vectorizers import ClassTfidfTransformer



In [ ]:
clean_articles_df = pd.read_parquet("clean_articles_step1.parquet")
clean_articles_df.shape
# clean_articles_df = pd.read_parquet("clean_articles_step1_(1).parquet")
# clean_articles_df.shape

In [ ]:
clean_articles_df.head()

In [ ]:
boilerplate_patterns = [
    r"skip\s*to\s*content\w*",
    r"watch\s*live\w*",
    r"community\s*calendar\w*",
    r"closings?(?:\s*&\s*|\s+and\s+)?delays?\b",
    r"submit\s*photos?(?:\s*&\s*|\s+and\s+)?videos?\b",
    r"download\s*(?:our\s*)?apps?\b",
    r"sign\s*up\s*for\s*(?:our\s*)?newsletters?\b",
    r"newsletters?\b",
    r"advertis\w*(?:\s+with\s+us)?\b",
    r"send\s*us\s*a\s*news\s*tip\b",
    r"weather\s*radar\b",
    r"election\s*results\b",
    r"home\s*news\s*weather\s*sports(?:\s*\w+){0,20}",
]
bp_regex = re.compile("|".join(boilerplate_patterns), flags=re.IGNORECASE)

def normalize_space(s: str) -> str:
    return re.sub(r"\s+", " ", str(s)).strip()

def topic_clean(title: str, content: str) -> str:
    t = normalize_space(title)
    c = normalize_space(content)

    # 去正文开头重复标题（最多两次）
    for _ in range(2):
        if t and c.lower().startswith(t.lower()):
            c = c[len(t):].strip(" -:|")

    # 删固定模板短语
    c = bp_regex.sub(" ", c)

    # 删导航词连续堆叠片段
    c = re.sub(
        r"(?:\b(home|news|weather|sports|live|video|search|menu|about|contact)\b[\s|/:,-]*){5,}",
        " ",
        c,
        flags=re.IGNORECASE,
    )

    c = re.sub(r"[\r\n\t]+", " ", c)
    c = re.sub(r"\s{2,}", " ", c).strip()
    return c

work_df = clean_articles_df.copy()
work_df["title"] = work_df["title"].fillna("").astype(str).str.strip()
work_df["content"] = work_df["content"].fillna("").astype(str).str.strip()

work_df["content_topic"] = [
    topic_clean(t, c) for t, c in zip(work_df["title"], work_df["content"])
]
work_df["text_raw"] = (work_df["title"] + " " + work_df["content_topic"]).str.strip()
work_df = work_df[work_df["text_raw"].str.len() > 0].reset_index(drop=True)

print("usable docs:", len(work_df))


# check = work_df["text_raw"].str.lower()
# for p in ["skip to content", "watch live", "community calendar", "advertise", "newsletter"]:
#     print(p, (check.str.contains(re.escape(p), regex=True)).mean())


In [ ]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(model_name)
tokenizer = embedder.tokenizer

MAX_TOKENS = 256  # test 先用 256

def truncate_text(text, max_tokens=256):
    ids = tokenizer.encode(
        text,
        add_special_tokens=True,
        truncation=True,
        max_length=max_tokens
    )
    return tokenizer.decode(ids, skip_special_tokens=True)

work_df["text_trunc"] = work_df["text_raw"].map(lambda x: truncate_text(x, MAX_TOKENS))


In [ ]:
TEST_N = 10000
RANDOM_SEED = 42

if len(work_df) > TEST_N:
    test_df = work_df.sample(n=TEST_N, random_state=RANDOM_SEED).reset_index(drop=True)
else:
    test_df = work_df.copy()

docs_test = test_df["text_trunc"].tolist()
print("test docs:", len(docs_test))


In [ ]:
# emb_test = embedder.encode(
#     docs_test,
#     batch_size=128,
#     show_progress_bar=True,
#     convert_to_numpy=True,
#     normalize_embeddings=True
# )

# print("embedding shape:", emb_test.shape)


In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

embedder = SentenceTransformer(model_name, device=device)
emb_test = embedder.encode(
    docs_test,
    batch_size=256 if device in ["cuda", "mps"] else 128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("device:", device)
print("embedding shape:", emb_test.shape)


In [ ]:
def eval_run(topics, emb):
    labels = np.array(topics)
    outlier_rate = np.mean(labels == -1)
    valid = labels != -1
    n_topics = len(set(labels[valid])) if valid.any() else 0

    sil = np.nan
    if valid.sum() > 20 and n_topics > 1:
        sil = silhouette_score(emb[valid], labels[valid])
        
    outlier_penalty = abs(outlier_rate - 0.225) * 2 if outlier_rate > 0.3 else abs(outlier_rate - 0.225)
    combined_score = (sil * 0.6) - outlier_penalty if not np.isnan(sil) else -outlier_penalty

    return {
        "n_topics": n_topics,
        "outlier_rate": float(outlier_rate),
        "silhouette": float(sil) if not np.isnan(sil) else np.nan,
        "combined_score": float(combined_score) if not np.isnan(combined_score) else np.nan
    }

param_grid = {
    "n_neighbors": [30, 35, 40, 45],         
    "min_dist": [0.05, 0.1, 0.15],          
    "min_cluster_size": [20, 25, 30],       
    "min_samples": [3, 5, 8]                
}

results = []
vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b"
)

total_runs = len(param_grid["n_neighbors"]) * len(param_grid["min_dist"]) * len(param_grid["min_cluster_size"]) * len(param_grid["min_samples"])
run_id = 0


for n_neighbors, min_dist, min_cluster_size, min_samples in product(
    param_grid["n_neighbors"],
    param_grid["min_dist"],
    param_grid["min_cluster_size"],
    param_grid["min_samples"]
):
    run_id += 1
    print(f"\n[{run_id}/{total_runs}] start -> nn={n_neighbors}, md={min_dist}, mcs={min_cluster_size}, ms={min_samples}")

    umap_model = UMAP(
        n_neighbors=n_neighbors,
        n_components=5,
        min_dist=min_dist,
        metric="cosine",
        random_state=42,
        low_memory=True
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=False
    )

    tm = BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        calculate_probabilities=False,
        verbose=False
    )

    # topics, _ = tm.fit_transform(docs_test, emb_test)
    # m = eval_run(topics, emb_test)
    # m.update({
    #     "n_neighbors": n_neighbors,
    #     "min_dist": min_dist,
    #     "min_cluster_size": min_cluster_size,
    #     "min_samples": min_samples
    # })
    # results.append(m)

    try:
        topics, _ = tm.fit_transform(docs_test, emb_test)
        m = eval_run(topics, emb_test)
        m.update({
            "n_neighbors": n_neighbors,
            "min_dist": min_dist,
            "min_cluster_size": min_cluster_size,
            "min_samples": min_samples
        })
        results.append(m)
        print(f"[{run_id}/{total_runs}] done  -> topics={m['n_topics']}, outlier={m['outlier_rate']:.4f}, sil={m['silhouette']:.4f}")
    except Exception as e:
        print(f"[{run_id}/{total_runs}] skip  -> {type(e).__name__}: {e}")
        continue


res_df = pd.DataFrame(results)

res_df['topic_range_score'] = res_df['n_topics'].apply(lambda x: 1 if 50 <= x <= 80 else 0.5 if 30 <= x < 50 or 80 < x <= 100 else 0)

res_df = res_df.sort_values(
    by=["combined_score", "topic_range_score", "silhouette"], 
    ascending=[False, False, False]
)




In [ ]:
res_df.head(20)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b"
)
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.1,
    metric="cosine",
    random_state=42,
    low_memory=True
)

hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=False
)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    calculate_probabilities=False,
    verbose=True
)

topics, _ = topic_model.fit_transform(docs_test, emb_test)
topic_info = topic_model.get_topic_info()
display(topic_info.head(20))


In [ ]:
doc_info = topic_model.get_document_info(docs_test)
display(doc_info[["Document", "Topic", "Name", "Probability"]].head(20))

# 看某个topic的样本文档
t = 0
sample_docs = doc_info[doc_info["Topic"] == t]["Document"].head(10).tolist()
for i, d in enumerate(sample_docs, 1):
    print(f"\n--- doc {i} ---\n{d[:500]}")


In [ ]:
fig = topic_model.visualize_barchart(top_n_topics=20)
fig.show()

In [ ]:
fig = topic_model.visualize_topics()
fig.show()